# Cleaning Example

This example demonstrates the deterministic cleaning operations performed by `df.dg.clean()` on a single DataFrame. The sample dataset intentionally includes inconsistent column names, leading and trailing whitespace, common missing-value representations, sparse rows and columns, empty rows and columns, and duplicate rows to illustrate the complete cleaning pipeline.

For detailed information about the cleaning API, see the [`clean accessor`](../accessor/clean.md) documentation.


In [1]:
import pandas as pd
# notice the need to danda import to have accessor df
import danda # noqa: F401  # registers the pandas accessor
from danda.report_renderer import ReportRenderer

renderer = ReportRenderer()

## DataFrame

The example DataFrame intentionally includes a variety of common data quality issues to demonstrate the full cleaning pipeline:

- Inconsistent column names
- Leading and trailing whitespace
- Common missing-value representations (for example, `"N/A"`, `"NULL"`, and `""`)
- Sparse rows
- Sparse columns
- A completely empty row
- A completely empty column
- Duplicate rows

In [2]:
df = pd.DataFrame({
    # RenameColumnsPlugin
    "First Name": [
        " Alice ",
        "Bob",
        "N/A",
        None,
        " Alice ",
        "",
        "Charlie",
        "Charlie",
    ],
    "First_Name": [
        "A",
        "B",
        "C",
        None,
        "A",
        "",
        "C",
        "C",
    ],
    "Last-Name": [
        " Smith ",
        " Jones",
        "Brown ",
        None,
        " Smith ",
        "",
        "White",
        "White",
    ],
    "Employee.ID": [
        101,
        102,
        103,
        None,
        101,
        None,
        104,
        104,
    ],

    # EmptySpacesPlugin + NormalizeMissingValuesPlugin
    "City": [
        " New York ",
        "London ",
        " N/A ",
        None,
        " New York ",
        " ",
        "Paris",
        "Paris",
    ],
    "Department": [
        " Sales ",
        "Engineering",
        "unknown",
        None,
        " Sales ",
        "NULL",
        "Finance",
        "Finance",
    ],

    # SparseColumnsPlugin (75% missing)
    "Mostly Missing": [
        None,
        None,
        None,
        1,
        None,
        None,
        None,
        None,
    ],

    # EmptyColumnsPlugin
    "All Missing": [
        None,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
    ],

    # Regular numeric column
    "Salary": [
        50000,
        60000,
        70000,
        None,
        50000,
        None,
        80000,
        80000,
    ],
})

# Make row 5 completely empty
df.loc[5] = pd.NA

# Make row 3 sparse (6/9 missing = 67%; adjust if needed)
df.loc[3, "Salary"] = 90000

# Make last row a duplicate of row 6
df.loc[7] = df.loc[6]

df

,First Name,First_Name,Last-Name,Employee.ID,City,Department,Mostly Missing,All Missing,Salary
0,Alice,A,Smith,101.0,New York,Sales,NaN,None,50000.0
1,Bob,B,Jones,102.0,London,Engineering,NaN,None,60000.0
2,N/A,C,Brown,103.0,N/A,unknown,NaN,None,70000.0
3,NaN,NaN,NaN,NaN,NaN,NaN,1.0,None,90000.0
4,Alice,A,Smith,101.0,New York,Sales,NaN,None,50000.0
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
6,Charlie,C,White,104.0,Paris,Finance,NaN,None,80000.0
7,Charlie,C,White,104.0,Paris,Finance,NaN,None,80000.0


## Enable the Rename Columns plugin

The Rename Columns plugin is disabled by default. Enable it before calling `df.dg.clean()` to automatically rename columns according to the configured naming convention.

In [3]:
# enable rename column plugin
df.dg.config.cleaning.rename_column_enabled = True

## clean data and show result

In [4]:
cleaned_df = df.dg.clean()
cleaned_df

,first_name,first_name_2,last_name,employee_id,city,department,mostly_missing,salary
0,Alice,A,Smith,101.0,New York,Sales,NaN,50000.0
1,Bob,B,Jones,102.0,London,Engineering,NaN,60000.0
2,N/A,C,Brown,103.0,N/A,unknown,NaN,70000.0
3,NaN,NaN,NaN,NaN,NaN,NaN,1.0,90000.0
6,Charlie,C,White,104.0,Paris,Finance,NaN,80000.0


## Show the cleaning report

The cleaning report summarizes the changes made by `df.dg.clean()`, including which plugins were applied and the modifications they performed.

In [6]:
# This will show what happen
print(renderer.render(cleaned_df.dg.report))

Danda Report

Clean
-----

RenameColumnsPlugin
    Renamed 9 column(s). columns: all_missing, city, department, employee_id, first_name, first_name_2, last_name, mostly_missing, salary

EmptySpacesPlugin
    Stripped leading/trailing whitespace from columns: first_name, last_name, city, department

EmptyRowsPlugin
    Number of deleted rows: 1

EmptyColumnsPlugin
    Number of deleted columns: 1

DropDuplicates
    Number of deleted rows: 2

Execution
---------
Plugins executed : 8
Plugin names     : RenameColumnsPlugin, EmptySpacesPlugin, NormalizeMissingValuesPlugin, SparseColumnsPlugin, SparseRowsPlugin, EmptyRowsPlugin, EmptyColumnsPlugin, DropDuplicates
Memory before    : 1,037 bytes
Memory after     : 459 bytes
Memory saved     : 578 bytes (55.7%)
